# 🗄️ Course 3 — Intermediate SQL

![SQL](https://img.shields.io/badge/Language-SQL-4479A1?logo=postgresql&logoColor=white)
![Platform](https://img.shields.io/badge/Platform-DataCamp-05192D?logo=datacamp&logoColor=white)
![Tool](https://img.shields.io/badge/Tool-PostgreSQL-336791?logo=postgresql&logoColor=white)
![Status](https://img.shields.io/badge/Status-Completed-brightgreen)

> **Platform:** DataCamp | **Track:** Data Analyst in Databricks
> **Tool:** PostgreSQL | **Database:** Films (title, budget, gross, release_year, country, certification, language) and People (name, birthdate, deathdate)

---

## 📋 About This Course

This course moves past a first SELECT statement into everything needed to actually interrogate a real dataset: counting and de-duplicating records, filtering with numeric, text, and NULL conditions, summarizing data with aggregate functions and arithmetic, and finally sorting and grouping results — including the difference between filtering individual rows (`WHERE`) and filtering aggregated groups (`HAVING`). It uses a **films** database (title, budget, gross, release year, country, certification, language) and a **people** database (name, birthdate, deathdate) throughout.

## 📚 Table of Contents

| Chapter | Topic |
|---------|-------|
| Chapter 1 | Querying a database: `COUNT()`, `DISTINCT`, query execution order, SQL style |
| Chapter 2 | Filtering records: comparison operators, `AND`/`OR`/`BETWEEN`, `LIKE`/`NOT LIKE`, `IN`, `NULL` |
| Chapter 3 | Aggregate functions: `AVG()`, `SUM()`, `MIN()`, `MAX()`, `ROUND()`, arithmetic |
| Chapter 4 | Sorting and grouping: `ORDER BY`, `GROUP BY`, `HAVING` |


## 📌 Chapter 1 — Querying a Database

---

### 1.1 — Counting and de-duplicating records

`COUNT()` and `DISTINCT` are the first tools for getting a feel for the shape of an unfamiliar table before writing any real analysis.

```sql
-- COUNT(field_name) only counts non-missing values in that field
SELECT COUNT(birthdate) AS count_birthdates
FROM people;

-- COUNT(*) counts every record, regardless of missing values
SELECT COUNT(*) AS total_records
FROM people;

-- DISTINCT removes duplicates from the result set
SELECT DISTINCT language
FROM films;

-- The two combine to count unique (non-null) values
SELECT COUNT(DISTINCT birthdate) AS count_distinct_birthdates
FROM people;
```

**Key distinction I want to remember:** `COUNT()` includes duplicates, `DISTINCT` excludes them — they answer two different questions ("how many rows have a value?" vs. "how many *unique* values are there?").

### 1.2 — Query execution order (and why it matters for debugging)

SQL isn't executed in the order it's written. A query like:

```sql
SELECT name
FROM people
LIMIT 10;
```

is actually processed roughly as `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT`. Knowing this explains two things that otherwise look like arbitrary rules: why aliases declared in `SELECT` can't be reused in a `WHERE` clause on the same query (WHERE runs before SELECT), and why most syntax errors — misspelled fields, missing commas, misspelled keywords like `SELCT` — point to the exact line but still require reading the *written* query carefully, since the engine's error message reflects where parsing broke, not necessarily execution order.

### 1.3 — Writing readable SQL

Formatting isn't required by the engine, but it's required for anyone (including future me) to maintain the query. The conventions I'm taking forward:
- Capitalize keywords (`SELECT`, `FROM`, `WHERE`), lowercase everything else.
- One clause per line for anything beyond a trivial query.
- Always end a statement with a semicolon — it marks the end of the query and makes translating between SQL flavors easier.
- Wrap non-standard field names (e.g. one containing a space, like `"release year"`) in double quotes.

Following a public style guide (e.g. [Holywell's SQL style guide](https://www.sqlstyle.guide/)) pays off directly in easier collaboration, debugging, and code that reads as professional rather than improvised.


## 📌 Chapter 2 — Filtering Records

---

### 2.1 — Filtering with comparison operators

`WHERE` filters rows against a condition, using the standard comparison set:

| Operator | Meaning |
|---|---|
| `>` | Greater than / after |
| `<` | Less than / before |
| `=` | Equal to |
| `>=` | Greater than or equal to |
| `<=` | Less than or equal to |
| `<>` | Not equal to |

```sql
SELECT title
FROM films
WHERE release_year > 1960;

-- Strings need single quotes
SELECT title
FROM films
WHERE country = 'Japan';
```

### 2.2 — Combining multiple conditions

`AND` requires every condition to hold; `OR` requires at least one. Both operands need to be full boolean expressions — `WHERE release_year = 1994 OR 2000` is invalid, it has to be `WHERE release_year = 1994 OR release_year = 2000`. Parentheses matter once `AND` and `OR` are mixed:

```sql
-- Films from 1994 or 1995, AND certified PG or R
SELECT title
FROM films
WHERE (release_year = 1994 OR release_year = 1995)
  AND (certification = 'PG' OR certification = 'R');

-- BETWEEN is inclusive shorthand for a >= ... AND <= ... range
SELECT title
FROM films
WHERE release_year BETWEEN 1994 AND 2000;
```

### 2.3 — Filtering text and lists: `LIKE`, `NOT LIKE`, `IN`

```sql
-- % matches zero, one, or many characters
SELECT name FROM people WHERE name LIKE 'Ade%';

-- _ matches exactly one character
SELECT name FROM people WHERE name LIKE 'Ev_';

-- NOT LIKE excludes a pattern
SELECT name FROM people WHERE name NOT LIKE 'A.%';

-- IN is a cleaner alternative to chained ORs on the same field
SELECT title FROM films WHERE release_year IN (1920, 1930, 1940);
```

### 2.4 — Handling `NULL`

`NULL` represents a missing value — human error, unavailable info, or simply unknown — and it behaves differently from any real value:

```sql
-- Rows where birthdate was never recorded
SELECT name
FROM people
WHERE birthdate IS NULL;

-- Rows where it WAS recorded
SELECT COUNT(name) AS count_birthdates
FROM people
WHERE birthdate IS NOT NULL;
```

**Takeaway:** `COUNT(field)` already silently excludes `NULL`s, so `COUNT(certification)` and `COUNT(certification) ... WHERE certification IS NOT NULL` return the same number — worth remembering before adding a redundant filter.


## 📌 Chapter 3 — Aggregate Functions

---

### 3.1 — The five core aggregates

```sql
SELECT AVG(budget) FROM films;   -- average
SELECT SUM(budget) FROM films;   -- total
SELECT MIN(budget) FROM films;   -- lowest value
SELECT MAX(budget) FROM films;   -- highest value
SELECT COUNT(budget) FROM films; -- non-null count
```

`AVG()` and `SUM()` only make sense on numeric fields, but `MIN()`/`MAX()`/`COUNT()` work on any data type — on text, `MIN()`/`MAX()` resolve alphabetically (`MIN(country)` returns "Afghanistan", the alphabetically-first value, not a "smallest" country in any other sense).

### 3.2 — Aggregating a filtered subset

`WHERE` can filter the rows *before* they're aggregated, which is the pattern for "average of X among Y":

```sql
SELECT ROUND(AVG(budget), 2) AS avg_budget
FROM films
WHERE release_year >= 2010;
```

`ROUND(number, decimal_places)` controls precision — a negative `decimal_places` rounds to the left of the decimal point (`ROUND(41072235, -5)` → `41100000`), which is handy for headline, order-of-magnitude figures.

### 3.3 — Arithmetic and integer division

```sql
SELECT (4 + 3);   -- 7
SELECT (4 / 3);   -- 1  (integer division: truncates)
SELECT (4.0 / 3.0); -- 1.333...  (float division: keeps the remainder)
```

**Gotcha worth keeping in mind:** dividing two integer columns truncates the result, so a ratio computed from integer fields may need an explicit cast (or a `.0` literal) to get a real decimal answer instead of a rounded-down whole number.

### 3.4 — Aliases live in `SELECT`, not before it

```sql
SELECT budget AS max_budget
FROM films
WHERE max_budget IS NOT NULL;  -- ❌ errors: alias not yet defined at WHERE time
```

This is a direct consequence of the execution order from Chapter 1: `WHERE` runs before `SELECT`, so an alias only exists for clauses that run *after* `SELECT` (like `ORDER BY`), never for `WHERE`.


## 📌 Chapter 4 — Sorting and Grouping

---

### 4.1 — `ORDER BY`

```sql
SELECT title, budget
FROM films
ORDER BY budget;        -- ascending by default

SELECT title, budget
FROM films
ORDER BY budget DESC;   -- descending

-- Multiple fields: the second field breaks ties in the first
SELECT title, wins, imdb_score
FROM best_movies
ORDER BY wins DESC, imdb_score DESC;
```

### 4.2 — `GROUP BY`

`GROUP BY` collapses rows sharing a value into a single summarized row — but every non-aggregated column in `SELECT` must also appear in `GROUP BY`, or the engine rejects the query:

```sql
-- Works: title only appears inside COUNT()
SELECT certification, COUNT(title) AS title_count
FROM films
GROUP BY certification;

-- Fails: title is selected raw but not grouped or aggregated
SELECT certification, title
FROM films
GROUP BY certification;

-- Multiple grouping fields, then sorted by the aggregate
SELECT certification, COUNT(title) AS title_count
FROM films
GROUP BY certification
ORDER BY title_count DESC;
```

### 4.3 — `HAVING`: filtering *groups*, not rows

This is the distinction I want to keep clearest from this whole course: `WHERE` filters individual records **before** grouping; `HAVING` filters the **already-aggregated groups**. Using `WHERE` with an aggregate function directly is a syntax error — `HAVING` exists specifically to fill that gap.

```sql
-- ❌ Invalid — WHERE can't reference an aggregate
SELECT release_year, COUNT(title) AS title_count
FROM films
GROUP BY release_year
WHERE COUNT(title) > 10;

-- ✅ Correct — HAVING filters the grouped result
SELECT release_year, COUNT(title) AS title_count
FROM films
GROUP BY release_year
HAVING COUNT(title) > 10;

-- Full pipeline combining everything from this course
SELECT certification, COUNT(title) AS title_count
FROM films
WHERE certification IN ('G', 'PG', 'PG-13')
GROUP BY certification
HAVING COUNT(title) > 500
ORDER BY title_count DESC
LIMIT 3;
```


## ✅ Skills demonstrated in this module

- Counting and de-duplicating records with `COUNT()` and `DISTINCT`, and explaining the difference between them.
- Reasoning about SQL's actual execution order (`FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT`) to debug alias errors and understand why certain clauses can't reference others.
- Writing readable, style-guide-consistent SQL (capitalization, line breaks, semicolons, quoting non-standard identifiers).
- Filtering records with comparison operators, `AND`/`OR`/`BETWEEN`, `LIKE`/`NOT LIKE` pattern matching, `IN`, and `NULL`-aware conditions (`IS NULL` / `IS NOT NULL`).
- Summarizing data with `AVG()`, `SUM()`, `MIN()`, `MAX()`, and `COUNT()`, including on filtered subsets and non-numeric fields.
- Using `ROUND()` for precision control and basic SQL arithmetic, including the integer-vs-float division gotcha.
- Sorting results with `ORDER BY` (single and multi-field, `ASC`/`DESC`).
- Grouping data with `GROUP BY`, and filtering aggregated groups correctly with `HAVING` instead of `WHERE`.
